# Notebook 4: Final comparison and error analysis

This notebook combines the locked zero-shot comparison with the three completed fine-tuning runs. It reports the final system table and produces per-job diagnostics for every system.

The fine-tuned E5 checkpoint was trained and evaluated without E5 `query:`/`passage:` prefixes, while the zero-shot E5 run used those prefixes. This notebook preserves that recorded protocol and marks the caveat in the exported metadata; E5 fine-tuning gains must therefore be interpreted as exploratory rather than a perfectly isolated prefix-controlled comparison.

Automated error analysis includes per-job ranking metrics, relevance AUC, core-versus-contextual AUC, top-10 false positives, and missed core skills. Qrels are treated as judgments, so an unjudged skill is not necessarily a genuinely incorrect recommendation.

In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
if not (ROOT / "ranking_metrics.py").exists():
    raise FileNotFoundError(f"Repository root not found from {Path.cwd()}")
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ranking_metrics import evaluate_query

RESULTS_DIR = ROOT / "results"
VAL_DIR = ROOT / "data" / "raw" / "TaskB" / "validation"
locked_path = RESULTS_DIR / "locked_model_comparison.csv"
fine_path = RESULTS_DIR / "finetuning_selected_models.csv"
if not locked_path.exists() or not fine_path.exists():
    raise FileNotFoundError("Run Notebooks 5 and 6 before this notebook")

locked = pd.read_csv(locked_path)
fine_tuned = pd.read_csv(fine_path)
print(f"zero-shot systems: {len(locked)}")
print(f"fine-tuned systems: {len(fine_tuned)}")

zero-shot systems: 5
fine-tuned systems: 3


## Final aggregate comparison

The final table joins the official locked zero-shot metrics with the recorded fine-tuned metrics. It does not recompute aggregate scores, preventing accidental changes to the completed experiments.

In [2]:
zero_summary = locked[["model_short_name", "model_name", "ndcg_graded", "ndcg_binary", "map_binary", "mrr_binary", "p@1", "p@5", "p@10"]].copy()
zero_summary["stage"] = "zero_shot"
zero_summary["protocol_note"] = "locked mean aliases; model-specific zero-shot prefixes"

fine_summary = fine_tuned[["model_short_name", "model_name", "ndcg_graded", "ndcg_binary", "map_binary", "mrr_binary", "p@1", "p@5", "p@10"]].copy()
fine_summary["stage"] = "fine_tuned"
fine_summary["protocol_note"] = "locked mean aliases; fine-tuning notebook used raw text for all models"

final_summary = pd.concat([zero_summary, fine_summary], ignore_index=True)
final_summary = final_summary.sort_values(["stage", "ndcg_graded"], ascending=[True, False]).reset_index(drop=True)
display(final_summary)
final_summary.to_csv(RESULTS_DIR / "final_system_comparison.csv", index=False)
print("Saved: results/final_system_comparison.csv")

,model_short_name,model_name,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10,stage,protocol_note
0,jobbert,TechWolf/JobBERT-v2,0.731888,0.781265,0.325751,0.799661,0.667763,0.588158,0.550658,fine_tuned,locked mean aliases; fine-tuning notebook used...
1,e5,intfloat/e5-base-v2,0.703989,0.749263,0.269418,0.724304,0.565789,0.513816,0.503947,fine_tuned,locked mean aliases; fine-tuning notebook used...
2,mpnet,all-mpnet-base-v2,0.698082,0.743776,0.260244,0.707061,0.562500,0.503289,0.476645,fine_tuned,locked mean aliases; fine-tuning notebook used...
3,jobbert,TechWolf/JobBERT-v2,0.738077,0.788888,0.341614,0.788649,0.648026,0.562500,0.517763,zero_shot,locked mean aliases; model-specific zero-shot ...
4,e5,intfloat/e5-base-v2,0.640110,0.682140,0.164249,0.695816,0.542763,0.450658,0.416776,zero_shot,locked mean aliases; model-specific zero-shot ...
5,mpnet,all-mpnet-base-v2,0.632662,0.676252,0.161164,0.630215,0.453947,0.398026,0.371053,zero_shot,locked mean aliases; model-specific zero-shot ...
6,bge,BAAI/bge-base-en-v1.5,0.630855,0.672003,0.155516,0.634001,0.444079,0.413816,0.373684,zero_shot,locked mean aliases; model-specific zero-shot ...
7,minilm,all-MiniLM-L6-v2,0.621283,0.664406,0.147637,0.634803,0.470395,0.383553,0.365461,zero_shot,locked mean aliases; model-specific zero-shot ...


Saved: results/final_system_comparison.csv


## Load the fixed development corpus and judgments

In [3]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t")
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t")
qrels = pd.read_csv(VAL_DIR / "qrels.tsv", sep="\t", header=None, names=["q_id", "iteration", "c_id", "relevance"])
corpus["skill_aliases"] = corpus["skill_aliases"].map(ast.literal_eval)
corpus["skill_aliases"] = corpus["skill_aliases"].map(lambda aliases: [str(alias).strip() for alias in aliases if str(alias).strip()])
skill_ids = corpus["c_id"].tolist()
skill_index = {skill_id: index for index, skill_id in enumerate(skill_ids)}
grades_by_job = {}
for job_id, group in qrels.groupby("q_id", sort=False):
    grades = np.zeros(len(skill_ids), dtype=np.float32)
    for row in group.itertuples(index=False):
        if row.c_id in skill_index:
            grades[skill_index[row.c_id]] = row.relevance
    grades_by_job[job_id] = grades
job_ids = [job_id for job_id in queries["q_id"] if job_id in grades_by_job]
query_texts = dict(zip(queries["q_id"], queries["jobtitle"]))
print(f"queries: {len(job_ids)}; candidate skills: {len(skill_ids)}")

queries: 304; candidate skills: 9052


## Per-job scoring helpers

Mean aliases are used for every system. Prefixes reproduce the completed runs: BGE and zero-shot E5 use their retrieval prefixes; fine-tuned checkpoints use raw text, matching Notebook 6.

In [4]:
MODEL_SPECS = {
    "minilm": {"name": "all-MiniLM-L6-v2", "path": "all-MiniLM-L6-v2", "query_prefix": "", "skill_prefix": ""},
    "mpnet": {"name": "all-mpnet-base-v2", "path": "all-mpnet-base-v2", "query_prefix": "", "skill_prefix": ""},
    "jobbert": {"name": "TechWolf/JobBERT-v2", "path": "models/jobbert_mnrl_mean_alias", "query_prefix": "", "skill_prefix": ""},
    "bge": {"name": "BAAI/bge-base-en-v1.5", "path": "BAAI/bge-base-en-v1.5", "query_prefix": "Represent this sentence for searching relevant skills: ", "skill_prefix": ""},
    "e5": {"name": "intfloat/e5-base-v2", "path": "models/e5_mnrl_mean_alias", "query_prefix": "", "skill_prefix": ""},
}
FINE_PATHS = {"jobbert": ROOT / "models" / "jobbert_mnrl_mean_alias", "e5": ROOT / "models" / "e5_mnrl_mean_alias", "mpnet": ROOT / "models" / "mpnet_mnrl_mean_alias"}

def encode(model, texts, prefix=""):
    return model.encode([prefix + str(text) for text in texts], batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)

def mean_aliases(model, prefix=""):
    aliases = corpus["skill_aliases"].tolist()
    counts = [len(items) for items in aliases]
    flat = encode(model, [alias for items in aliases for alias in items], prefix)
    vectors, start = [], 0
    for count in counts:
        vector = flat[start:start + count].mean(axis=0)
        vector /= max(np.linalg.norm(vector), 1e-12)
        vectors.append(vector)
        start += count
    return np.asarray(vectors, dtype=np.float32)

def score_system(model, query_prefix="", skill_prefix=""):
    jobs = encode(model, [query_texts[job_id] for job_id in job_ids], query_prefix)
    skills = mean_aliases(model, skill_prefix)
    return jobs @ skills.T

def per_job_rows(scores, system_name):
    rows = []
    for index, job_id in enumerate(job_ids):
        grades = grades_by_job[job_id]
        metrics = evaluate_query(scores[index], grades)
        relevant = grades > 0
        core_context = grades > 0
        binary_auc = roc_auc_score(relevant.astype(int), scores[index]) if relevant.any() and (~relevant).any() else np.nan
        core_auc = roc_auc_score((grades == 2).astype(int), scores[index]) if (grades == 2).any() and (grades != 2).any() else np.nan
        rows.append({"system": system_name, "job_id": job_id, "job_title": query_texts[job_id], "n_relevant": int(relevant.sum()), "n_core": int((grades == 2).sum()), "relevance_auc": binary_auc, "core_auc": core_auc, **metrics})
    return rows

## Run error analysis for zero-shot and fine-tuned systems

This cell is computationally expensive because it encodes the complete candidate corpus for each system. It reuses the saved fine-tuned checkpoints and writes per-job results incrementally after every system.

In [5]:
analysis_rows = []
system_specs = [
    ("jobbert_zero_shot", "TechWolf/JobBERT-v2", "", ""),
    ("e5_zero_shot", "intfloat/e5-base-v2", "query: ", "passage: "),
    ("mpnet_zero_shot", "all-mpnet-base-v2", "", ""),
    ("minilm_zero_shot", "all-MiniLM-L6-v2", "", ""),
    ("bge_zero_shot", "BAAI/bge-base-en-v1.5", "Represent this sentence for searching relevant skills: ", ""),
    ("jobbert_fine_tuned", str(FINE_PATHS["jobbert"]), "", ""),
    ("e5_fine_tuned", str(FINE_PATHS["e5"]), "", ""),
    ("mpnet_fine_tuned", str(FINE_PATHS["mpnet"]), "", ""),
]
for system_name, model_path, query_prefix, skill_prefix in system_specs:
    print(f"\n===== {system_name} =====")
    model = SentenceTransformer(model_path)
    scores = score_system(model, query_prefix, skill_prefix)
    analysis_rows.extend(per_job_rows(scores, system_name))
    pd.DataFrame(analysis_rows).to_csv(RESULTS_DIR / "error_analysis_per_job.csv", index=False)
    del model
per_job = pd.DataFrame(analysis_rows)
print(f"Saved {len(per_job)} per-job rows")


===== jobbert_zero_shot =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1006 [00:00<?, ?it/s]


===== e5_zero_shot =====


...


===== mpnet_fine_tuned =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/1006 [00:00<?, ?it/s]

Saved 2432 per-job rows


## Error-analysis summaries and difficult jobs

In [6]:
summary = per_job.groupby("system", as_index=False).agg({"ndcg_graded": "mean", "ndcg_binary": "mean", "map_binary": "mean", "mrr_binary": "mean", "p@1": "mean", "p@5": "mean", "p@10": "mean", "relevance_auc": "mean", "core_auc": "mean"}).sort_values("ndcg_graded", ascending=False)
display(summary)
summary.to_csv(RESULTS_DIR / "error_analysis_summary.csv", index=False)

hardest = per_job.sort_values(["system", "ndcg_graded"]).groupby("system", as_index=False).head(10)
hardest.to_csv(RESULTS_DIR / "error_analysis_hardest_jobs.csv", index=False)
display(hardest[["system", "job_id", "job_title", "ndcg_graded", "map_binary", "relevance_auc", "core_auc"]].head(30))
print("Saved error-analysis summary and hardest-job files")

,system,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10,relevance_auc,core_auc
4,jobbert_zero_shot,0.738129,0.788926,0.341624,0.790293,0.651316,0.563158,0.517763,0.959570,0.958463
3,jobbert_fine_tuned,0.731888,0.781265,0.325751,0.799661,0.667763,0.588158,0.550658,0.941952,0.948311
1,e5_fine_tuned,0.703989,0.749263,0.269418,0.724304,0.565789,0.513816,0.503947,0.921882,0.944478
6,mpnet_fine_tuned,0.698082,0.743776,0.260244,0.707061,0.562500,0.503289,0.476645,0.922192,0.943593
2,e5_zero_shot,0.640146,0.682171,0.164263,0.696617,0.542763,0.450658,0.416776,0.851680,0.877232
7,mpnet_zero_shot,0.632664,0.676263,0.161173,0.630035,0.453947,0.399342,0.371053,0.854275,0.873187
0,bge_zero_shot,0.630851,0.672005,0.155514,0.633949,0.444079,0.413158,0.373684,0.839031,0.874369
5,minilm_zero_shot,0.621388,0.664475,0.147649,0.638562,0.476974,0.383553,0.365789,0.830314,0.853982


,system,job_id,job_title,ndcg_graded,map_binary,relevance_auc,core_auc
1386,bge_zero_shot,dev_qb_jt_171,gl specialist,0.330592,0.012401,0.624369,0.606031
1415,bge_zero_shot,dev_qb_jt_200,post closer,0.384028,0.017515,0.726673,0.768514
1267,bge_zero_shot,dev_qb_jt_52,wrapper operator,0.401932,0.019114,0.666597,0.766516
1265,bge_zero_shot,dev_qb_jt_50,seamer operator,0.431668,0.045018,0.850381,0.850364
1250,bge_zero_shot,dev_qb_jt_35,second pressman,0.439915,0.030085,0.706624,0.674778
1416,bge_zero_shot,dev_qb_jt_201,loan operations assistant,0.468353,0.040829,0.770964,0.823057
1418,bge_zero_shot,dev_qb_jt_203,escrow closing specialist,0.472821,0.038835,0.769937,0.791144
1422,bge_zero_shot,dev_qb_jt_207,associate merchandise planner,0.475388,0.051393,0.726998,0.696558
1427,bge_zero_shot,dev_qb_jt_212,identity and access management administrator,0.477735,0.044492,0.752447,0.827417
1305,bge_zero_shot,dev_qb_jt_90,accounting cashier,0.481674,0.064912,0.841289,0.904393


Saved error-analysis summary and hardest-job files
